In [1]:
import requests
import json
from getpass import getpass
import os
from dotenv import load_dotenv
import time

# Configuration
load_dotenv()
api_key = os.getenv("RUNPOD_API_KEY")
if not api_key:
    api_key = getpass("Enter your RunPod API key: ")

endpoint_id = "a48mrbdsbzg35n"

run_url = f"https://api.runpod.ai/v2/{endpoint_id}/run"
status_url_template = f"https://api.runpod.ai/v2/{endpoint_id}/status/"

# Test each model type to see what's available
test_workflows = {
    "checkpoints": {
        "input": {
            "workflow": {
                "1": {"inputs": {"ckpt_name": "test.safetensors"}, "class_type": "CheckpointLoaderSimple"}
            }
        }
    },
    "unet": {
        "input": {
            "workflow": {
                "1": {"inputs": {"unet_name": "test.safetensors", "weight_dtype": "default"}, "class_type": "UNETLoader"}
            }
        }
    },
    "clip": {
        "input": {
            "workflow": {
                "1": {"inputs": {"clip_name": "test.safetensors", "type": "qwen_image", "device": "default"}, "class_type": "CLIPLoader"}
            }
        }
    },
    "vae": {
        "input": {
            "workflow": {
                "1": {"inputs": {"vae_name": "test.safetensors"}, "class_type": "VAELoader"}
            }
        }
    },
    "loras": {
        "input": {
            "workflow": {
                "1": {"inputs": {"lora_01": "test.safetensors", "strength_01": 1.0, "lora_02": "None", "strength_02": 1.0, "lora_03": "None", "strength_03": 1.0, "lora_04": "None", "strength_04": 1.0, "model": ["2", 0], "clip": ["2", 1]}, "class_type": "Lora Loader Stack (rgthree)"},
                "2": {"inputs": {"ckpt_name": "test.safetensors"}, "class_type": "CheckpointLoaderSimple"}
            }
        }
    }
}

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

print("=" * 70)
print("RUNPOD WORKER MODEL AVAILABILITY CHECK")
print("=" * 70)

for model_type, workflow in test_workflows.items():
    print(f"\n🔍 Checking {model_type.upper()}...")
    
    response = requests.post(run_url, headers=headers, json=workflow)
    
    if response.status_code == 200:
        response_data = response.json()
        job_id = response_data.get('id')
        
        if job_id:
            status_url = status_url_template + job_id
            
            # Wait for response
            for attempt in range(15):
                time.sleep(2)
                status_response = requests.get(status_url, headers=headers)
                status_data = status_response.json()
                
                if status_data.get('status') in ['COMPLETED', 'FAILED']:
                    if 'error' in status_data:
                        error_msg = status_data['error']
                        
                        # Extract available models from error message
                        if "not in [" in error_msg:
                            start = error_msg.find("not in [") + 8
                            end = error_msg.find("]", start)
                            models_str = error_msg[start:end]
                            models = [m.strip().strip("'") for m in models_str.split(",") if m.strip() and m.strip() != "''"]
                            
                            if models and models != ['']:
                                print(f"  ✅ Found {len(models)} model(s):")
                                for model in models[:10]:  # Show first 10
                                    print(f"     • {model}")
                                if len(models) > 10:
                                    print(f"     ... and {len(models) - 10} more")
                            else:
                                print(f"  ❌ No models found")
                        else:
                            print(f"  ⚠️  Unexpected error format")
                    break
            else:
                print(f"  ⏱️  Timeout waiting for response")
    else:
        print(f"  ❌ API request failed: {response.status_code}")
    
    time.sleep(1)  # Rate limiting

print("\n" + "=" * 70)
print("\n📋 SUMMARY:")
print("If models are missing, you need to:")
print("1. Configure your RunPod worker to sync from S3")
print("2. Or manually copy models to the worker")
print("3. Or use a different worker template with models pre-installed")
print("=" * 70)

Enter your RunPod API key:  ········


RUNPOD WORKER MODEL AVAILABILITY CHECK

🔍 Checking CHECKPOINTS...
  ⚠️  Unexpected error format

🔍 Checking UNET...
  ⚠️  Unexpected error format

🔍 Checking CLIP...
  ⚠️  Unexpected error format

🔍 Checking VAE...
  ⚠️  Unexpected error format

🔍 Checking LORAS...
  ⚠️  Unexpected error format


📋 SUMMARY:
If models are missing, you need to:
1. Configure your RunPod worker to sync from S3
2. Or manually copy models to the worker
3. Or use a different worker template with models pre-installed
